In [1]:
# ============================================================
# 셀 1: 설치 및 환경 설정
# ============================================================
# LunarLander-v2는 Box2D 물리엔진 기반이라 별도 설치 필요
# gymnasium: OpenAI Gym의 최신 버전 (구 gym과 API 거의 동일)
# ============================================================
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import threading
import time
import matplotlib.pyplot as plt
from collections import deque

# ── 재현성을 위한 시드 고정 ──────────────────────────────────
SEED = 2026
ENV_NAME = "LunarLander-v3"
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── 디바이스 설정 ────────────────────────────────────────────
# A3C는 Worker들이 CPU에서 병렬로 환경을 돌리고
# gradient를 Global Network(GPU or CPU)에 전달하는 구조
# Colab 무료 환경에서는 CPU로도 충분히 동작함
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 환경 스펙 확인 ───────────────────────────────────────────
env = gym.make(ENV_NAME)
STATE_DIM = env.observation_space.shape[0]   # 8
ACTION_DIM = env.action_space.n               # 4
env.close()

print(f"State  dim : {STATE_DIM}")
print(f"Action dim : {ACTION_DIM}")

# ── 하이퍼파라미터 (두 알고리즘 공통) ────────────────────────
GAMMA = 0.99    # 할인율
LR = 3e-4    # 학습률
MAX_EPISODES = 2000    # 총 학습 에피소드 수

# A3C 전용
N_WORKERS = 4       # Worker 스레드 수
T_MAX = 5       # n-step rollout 길이
ENTROPY_BETA = 0.01    # 탐색 촉진을 위한 엔트로피 보너스 계수

Using device: cuda
State  dim : 8
Action dim : 4


In [2]:
# ============================================================
# 셀 2: 공통 ActorCritic 네트워크 정의
# ============================================================
# REINFORCE와 A3C 둘 다 이 네트워크를 사용
# 구조: 공유 Backbone → Actor head / Critic head 분리
#
#   Input (8)
#     │
#   [Shared Backbone]
#   Linear(8→256) → ReLU
#   Linear(256→128) → ReLU
#     │
#   ┌─┴─────────────┐
# [Actor head]   [Critic head]
# Linear(128→4)  Linear(128→1)
# Softmax↓            │
# π(a|s)           V(s)
# ============================================================

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()

        # 공유 Backbone: 두 head가 같은 특징을 학습
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        # Actor head: 각 행동의 확률 출력
        self.actor_head = nn.Linear(128, action_dim)

        # Critic head: 현재 상태의 가치 출력 (스칼라)
        self.critic_head = nn.Linear(128, 1)

        # 가중치 초기화 (학습 안정성 향상)
        self._initialize_weights()

    def _initialize_weights(self):
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = self.backbone(x)

        # Actor: softmax로 확률 분포 변환
        policy = F.softmax(self.actor_head(x), dim=-1)

        # Critic: 상태 가치 (squeeze로 스칼라화)
        value = self.critic_head(x).squeeze(-1)

        return policy, value

    def get_action(self, state):
        # 네트워크 파라미터가 있는 device를 동적으로 참조
        # → REINFORCE(CUDA), Worker local_net(CPU) 둘 다 자동 대응
        device = next(self.parameters()).device
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        policy, value = self.forward(state_t)
    
        dist = torch.distributions.Categorical(policy)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
    
        return action.item(), log_prob, value.squeeze(0), entropy


# ── 네트워크 구조 확인 ───────────────────────────────────────
net = ActorCritic(STATE_DIM, ACTION_DIM)
print(net)
print(f"\n총 파라미터 수: {sum(p.numel() for p in net.parameters()):,}")

# 동작 테스트
dummy_state = np.zeros(STATE_DIM)
action, log_prob, value, entropy = net.get_action(dummy_state)
print(f"\n[동작 테스트]")
print(f"  action   : {action}")
print(f"  log_prob : {log_prob.item():.4f}")
print(f"  value    : {value.item():.4f}")
print(f"  entropy  : {entropy.item():.4f}")

ActorCritic(
  (backbone): Sequential(
    (0): Linear(in_features=8, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
  )
  (actor_head): Linear(in_features=128, out_features=4, bias=True)
  (critic_head): Linear(in_features=128, out_features=1, bias=True)
)

총 파라미터 수: 35,845

[동작 테스트]
  action   : 0
  log_prob : -1.3863
  value    : 0.0000
  entropy  : 1.3863


In [3]:
# ============================================================
# 셀 3: REINFORCE + Baseline 구현
# ============================================================
# 업데이트 흐름:
#   1. episode 끝까지 진행하며 (s, a, r, log_prob, value) 저장
#   2. 끝에서부터 Monte Carlo return G_t 계산
#   3. Advantage = G_t - V(s) (baseline으로 분산 감소)
#   4. Actor loss  = -log_prob * Advantage
#      Critic loss = MSE(V(s), G_t)
#      Total loss  = actor_loss + critic_loss
#   5. episode 단위로 한 번에 업데이트
# ============================================================

def compute_returns(rewards, gamma):
    """
    episode 끝에서부터 역순으로 G_t 계산
    G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...
    """
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.FloatTensor(returns)

    return returns


def train_reinforce(max_episodes=MAX_EPISODES, seed=SEED):
    env = gym.make(ENV_NAME)
    net = ActorCritic(STATE_DIM, ACTION_DIM).to(device)
    optimizer = optim.Adam(net.parameters(), lr=LR)

    episode_rewards = []   # 에피소드별 총 리워드 기록
    recent_rewards = deque(maxlen=100)  # 최근 100 에피소드 이동 평균용

    start_time = time.time()

    for episode in range(max_episodes):
        state, _ = env.reset(seed=seed + episode)

        # ── 에피소드 진행하며 trajectory 수집 ──────────────────
        log_probs = []
        values = []
        rewards = []

        done = False
        while not done:
            action, log_prob, value, _ = net.get_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)

            state = next_state

        # ── Monte Carlo return 계산 ────────────────────────────
        returns = compute_returns(rewards, GAMMA).to(device)
        returns_norm = (returns - returns.mean()) / (returns.std() + 1e-8)  # returns 자체를 정규화
        log_probs = torch.stack(log_probs)
        values    = torch.stack(values)

        # ── Advantage 계산 ────────────────────────────────────
        # detach(): Advantage는 상수 취급, Critic 학습에 영향 X
        advantage = returns_norm - values.detach()  # 별도 advantage 정규화 불필요

        # ── Loss 계산 ─────────────────────────────────────────
        actor_loss  = -(log_probs * advantage).mean()
        critic_loss = 0.5 * F.mse_loss(values, returns_norm)  # 동일한 스케일 사용
        loss = actor_loss + critic_loss

        # ── 업데이트 ──────────────────────────────────────────
        optimizer.zero_grad()
        loss.backward()
        # gradient clipping: 너무 큰 gradient로 인한 발산 방지
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
        optimizer.step()

        # ── 기록 ──────────────────────────────────────────────
        total_reward = sum(rewards)
        episode_rewards.append(total_reward)
        recent_rewards.append(total_reward)

        if (episode + 1) % 100 == 0:
            elapsed = time.time() - start_time
            print(f"[REINFORCE] EP {episode+1:4d} | "
                  f"avg(100): {np.mean(recent_rewards):7.2f} | "
                  f"time: {elapsed:.1f}s")

    env.close()
    elapsed_total = time.time() - start_time
    print(f"\n[REINFORCE] 학습 완료 | 총 시간: {elapsed_total:.1f}s")

    return episode_rewards, elapsed_total

In [4]:
# ============================================================
# 셀 4: A3C 구현 (수정 버전)
# ============================================================
# 수정 사항:
#   1. global_net을 CPU에 유지 (CUDA + threading = 커널 크래시)
#   2. Worker.run() 구조 수정:
#      - 기존: T_MAX 스텝마다 env.reset() → 에피소드 대부분 버림
#      - 수정: 에피소드를 유지하면서 T_MAX 스텝마다 gradient update
#      (이것이 원래 A3C 논문의 올바른 구조)
# ============================================================

class Worker(threading.Thread):
    def __init__(self, worker_id, global_net, optimizer,
                 episode_rewards, lock, counter, max_episodes):
        super(Worker, self).__init__()
        self.worker_id       = worker_id
        self.global_net      = global_net
        self.optimizer       = optimizer
        self.episode_rewards = episode_rewards
        self.lock            = lock
        self.counter         = counter
        self.max_episodes    = max_episodes

        self.local_net = ActorCritic(STATE_DIM, ACTION_DIM)
        self.env       = gym.make(ENV_NAME)

    def sync_with_global(self):
        """Global net → Local net 파라미터 동기화"""
        # global_net이 CPU에 있으므로 .cpu() 변환 불필요
        with self.lock:
            self.local_net.load_state_dict(self.global_net.state_dict())

    def compute_n_step_returns(self, rewards, last_value, dones):
        """n-step return 역순 계산"""
        G = last_value
        returns = []
        for r, done in zip(reversed(rewards), reversed(dones)):
            G = r + GAMMA * G * (1 - done)
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        return returns

    def run(self):
        # ── 에피소드 시작 (outer loop 바깥에서 상태 유지) ────
        state, _ = self.env.reset(seed=self.worker_id * 1000)
        episode_reward = 0

        while True:
            # ── 종료 조건 확인 ────────────────────────────────
            with self.lock:
                if self.counter[0] >= self.max_episodes:
                    break

            # ── Global → Local 동기화 ────────────────────────
            self.sync_with_global()

            # ── T_MAX 스텝 rollout 수집 ──────────────────────
            log_probs = []
            values    = []
            rewards   = []
            entropies = []
            dones     = []

            done = False
            for _ in range(T_MAX):
                action, log_prob, value, entropy = self.local_net.get_action(state)
                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                log_probs.append(log_prob)
                values.append(value)
                rewards.append(reward)
                entropies.append(entropy)
                dones.append(float(done))

                episode_reward += reward
                state = next_state

                if done:
                    break

            # ── Bootstrap: 에피소드 안 끝났으면 V(s') 추정 ───
            if done:
                last_value = 0.0
            else:
                with torch.no_grad():
                    _, last_value = self.local_net.forward(
                        torch.FloatTensor(state).unsqueeze(0)
                    )
                    last_value = last_value.item()

            # ── Loss 계산 ────────────────────────────────────
            returns   = self.compute_n_step_returns(rewards, last_value, dones)
            log_probs = torch.stack(log_probs)
            values    = torch.stack(values)
            entropies = torch.stack(entropies)
            advantage = returns - values.detach()

            actor_loss   = -(log_probs * advantage).mean()
            critic_loss  = F.mse_loss(values, returns)
            entropy_loss = entropies.mean()
            loss = actor_loss + critic_loss - ENTROPY_BETA * entropy_loss

            # ── Local gradient → Global net에 적용 ───────────
            with self.lock:
                self.optimizer.zero_grad()
                loss.backward()
                for local_param, global_param in zip(
                    self.local_net.parameters(), self.global_net.parameters()
                ):
                    if local_param.grad is not None:
                        global_param.grad = local_param.grad.clone()
                nn.utils.clip_grad_norm_(self.global_net.parameters(), max_norm=0.5)
                self.optimizer.step()

            # ── 에피소드 종료 처리 ───────────────────────────
            if done:
                with self.lock:
                    self.counter[0] += 1
                    self.episode_rewards.append(episode_reward)
                    ep = self.counter[0]

                if ep % 100 == 0:
                    recent = self.episode_rewards[-100:]
                    print(f"[A3C W{self.worker_id}] EP {ep:4d} | "
                          f"avg(100): {np.mean(recent):7.2f}")

                # ── 새 에피소드 시작 ─────────────────────────
                state, _ = self.env.reset()
                episode_reward = 0


def train_a3c(max_episodes=MAX_EPISODES):
    # ★ 수정: CPU에 유지 (threading + CUDA = 크래시 원인)
    global_net = ActorCritic(STATE_DIM, ACTION_DIM)  # CPU
    optimizer  = optim.Adam(global_net.parameters(), lr=LR)

    episode_rewards = []
    lock    = threading.Lock()
    counter = [0]

    start_time = time.time()

    workers = [
        Worker(i, global_net, optimizer,
               episode_rewards, lock, counter, max_episodes)
        for i in range(N_WORKERS)
    ]
    for w in workers:
        w.start()
    for w in workers:
        w.join()

    for w in workers:
        w.env.close()

    elapsed_total = time.time() - start_time
    print(f"\n[A3C] 학습 완료 | 총 시간: {elapsed_total:.1f}s")

    return episode_rewards, elapsed_total


In [ ]:
# ============================================================
# 셀 5: 학습 실행
# ============================================================
# REINFORCE와 A3C를 동일한 조건으로 실행
# - 동일한 MAX_EPISODES
# - 동일한 SEED
# - 결과를 딕셔너리로 저장 → 셀 6 시각화에서 사용
# ============================================================

results = {}

# ── REINFORCE 실행 ───────────────────────────────────────────
print("=" * 55)
print("REINFORCE + Baseline 학습 시작")
print("=" * 55)
reinforce_rewards, reinforce_time = train_reinforce(
    max_episodes=MAX_EPISODES,
    seed=SEED
)
results["REINFORCE"] = {
    "rewards" : reinforce_rewards,
    "time"    : reinforce_time
}
print()

# ── A3C 실행 ─────────────────────────────────────────────────
print("=" * 55)
print(f"A3C 학습 시작 (Worker {N_WORKERS}개)")
print("=" * 55)
a3c_rewards, a3c_time = train_a3c(
    max_episodes=MAX_EPISODES
)
results["A3C"] = {
    "rewards" : a3c_rewards,
    "time"    : a3c_time
}

# ── 요약 출력 ────────────────────────────────────────────────
print()
print("=" * 55)
print("학습 결과 요약")
print("=" * 55)

for algo, data in results.items():
    rewards = data["rewards"]
    last100 = rewards[-100:]
    print(f"\n[{algo}]")
    print(f"  총 학습 시간       : {data['time']:.1f}s")
    print(f"  최종 100ep 평균    : {np.mean(last100):.2f}")
    print(f"  최종 100ep 표준편차: {np.std(last100):.2f}")
    print(f"  최고 에피소드 리워드: {max(rewards):.2f}")

    # 수렴 기준: 이동평균 200 이상 첫 도달 에피소드
    window = 100
    converged_ep = None
    for i in range(window, len(rewards)):
        if np.mean(rewards[i-window:i]) >= 200:
            converged_ep = i
            break
    if converged_ep:
        print(f"  수렴 에피소드 (avg200 달성): {converged_ep}")
    else:
        print(f"  수렴 에피소드 (avg200 달성): 미달성")

REINFORCE + Baseline 학습 시작
[REINFORCE] EP  100 | avg(100): -185.88 | time: 63.5s


In [ ]:
# ============================================================
# 셀 6: 비교 시각화
# ============================================================
# 총 3개 그래프:
#   1. 에피소드별 리워드 곡선 (raw + 이동평균)
#   2. 이동평균 수렴 비교
#   3. 실험 지표 요약 bar chart
# ============================================================

def moving_average(data, window=100):
    ma = []
    for i in range(len(data)):
        start = max(0, i - window + 1)
        ma.append(np.mean(data[start:i+1]))
    return np.array(ma)


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("REINFORCE vs A3C on LunarLander-v3", fontsize=16, fontweight="bold")

colors = {"REINFORCE": "#4C72B0", "A3C": "#DD8452"}

# ── 그래프 1: Raw 리워드 곡선 ─────────────────────────────────
ax1 = axes[0, 0]
for algo, data in results.items():
    rewards = data["rewards"]
    ax1.plot(rewards, alpha=0.3, color=colors[algo], linewidth=0.8)
    ax1.plot(moving_average(rewards), color=colors[algo],
             linewidth=2.0, label=f"{algo} (MA100)")

ax1.axhline(y=200, color="red", linestyle="--", linewidth=1.2, label="수렴 기준 (200)")
ax1.set_title("에피소드별 리워드 (Raw + 이동평균)")
ax1.set_xlabel("Episode")
ax1.set_ylabel("Total Reward")
ax1.legend()
ax1.grid(alpha=0.3)

# ── 그래프 2: 이동평균만 확대 비교 ───────────────────────────
ax2 = axes[0, 1]
for algo, data in results.items():
    ma = moving_average(data["rewards"])
    ax2.plot(ma, color=colors[algo], linewidth=2.0, label=algo)

ax2.axhline(y=200, color="red", linestyle="--", linewidth=1.2, label="수렴 기준 (200)")
ax2.set_title("이동평균 수렴 비교 (MA100)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Moving Average Reward")
ax2.legend()
ax2.grid(alpha=0.3)

# ── 그래프 3: 리워드 분포 boxplot ─────────────────────────────
ax3 = axes[1, 0]
last100_data = [results[algo]["rewards"][-100:] for algo in results]
bp = ax3.boxplot(last100_data, labels=list(results.keys()),
                 patch_artist=True, notch=True)
for patch, algo in zip(bp["boxes"], results.keys()):
    patch.set_facecolor(colors[algo])
    patch.set_alpha(0.7)

ax3.axhline(y=200, color="red", linestyle="--", linewidth=1.2, label="수렴 기준 (200)")
ax3.set_title("최종 100 에피소드 리워드 분포")
ax3.set_ylabel("Total Reward")
ax3.legend()
ax3.grid(alpha=0.3)

# ── 그래프 4: 지표 요약 bar chart ────────────────────────────
ax4 = axes[1, 1]

metrics = {
    "최종 100ep\n평균":  [np.mean(results[a]["rewards"][-100:]) for a in results],
    "최종 100ep\n표준편차": [np.std(results[a]["rewards"][-100:]) for a in results],
    "학습 시간(s)": [results[a]["time"] for a in results],
}

x     = np.arange(len(metrics))
width = 0.3
algos = list(results.keys())

for i, algo in enumerate(algos):
    vals = [list(metrics.values())[j][i] for j in range(len(metrics))]
    bars = ax4.bar(x + i * width, vals, width,
                   label=algo, color=colors[algo], alpha=0.8)
    for bar, val in zip(bars, vals):
        ax4.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1,
                 f"{val:.1f}", ha="center", va="bottom", fontsize=9)

ax4.set_title("실험 지표 요약")
ax4.set_xticks(x + width / 2)
ax4.set_xticklabels(list(metrics.keys()))
ax4.legend()
ax4.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("reinforce_vs_a3c.png", dpi=150, bbox_inches="tight")
plt.show()
print("그래프 저장 완료: reinforce_vs_a3c.png")